# AML adapter for scGPT: train, benchmark, use

Runs the `amladapter` package end to end on one Colab GPU:

1. downloads van Galen 2019 (GSE116256), Naldini 2023 (GSE185991) and the scGPT whole-human checkpoint;
2. trains AML adapters (self-supervised and supervised) on a frozen scGPT;
3. scores them against HVG/PCA baselines and frozen scGPT on three patient-held-out tasks
   (T1 malignant vs normal; **T2 van Galen → Naldini genotype transfer**; T3 21-state annotation);
4. applies the pre-specified **PROMISE** rule and ships the model chosen on van Galen cross-validation;
5. loads the shipped model and predicts on new cells.

**Runtime → Change runtime type → T4 or L4 GPU.** The full run takes roughly 4–5 h on a T4. Use `TASKS`/`OVERRIDES`
below to run less. Everything is stored on Google Drive, so a disconnect loses at most the step in progress
(downloads and data caches resume; rerun the benchmark cell to restart the benchmark itself).

In [ ]:
#@title 1 · GPU and Drive
import subprocess, sys, os
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout or 'NO GPU: switch the runtime to a GPU')
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/aml_adapter'      # data, scGPT weights and results live here
os.makedirs(WORK, exist_ok=True)

In [ ]:
#@title 2 · Get the code
# Default: clone the repository branch. If the repository is private, upload the `aml-adapter` folder to
# Drive at WORK/aml-adapter instead and set USE_DRIVE_COPY = True.
REPO = 'https://github.com/dhillonv10/dhillonv10.github.io.git'
BRANCH = 'claude/nifty-volta-t6qfeq'
USE_DRIVE_COPY = False
CODE = f'{WORK}/aml-adapter' if USE_DRIVE_COPY else '/content/repo/aml-adapter'
if not USE_DRIVE_COPY and not os.path.exists(CODE):
    r = subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO, '/content/repo'], capture_output=True, text=True)
    print(r.stdout, r.stderr)
assert os.path.exists(f'{CODE}/amladapter/benchmark.py'), 'code not found: see the note above'
os.chdir(CODE); sys.path.insert(0, CODE)
print('code at', CODE)

In [ ]:
#@title 3 · Install dependencies (~2-3 min)
%pip -q install -r requirements.txt

In [ ]:
#@title 4 · Optional: 2-minute smoke test (synthetic data, tiny random backbone; checks the install, not the science)
RUN_SMOKE = True
if RUN_SMOKE:
    !python scripts/run_benchmark.py --profile smoke --out /content/smoke 2>&1 | grep -vi warn | tail -5

In [ ]:
#@title 5 · Run the benchmark (downloads ~2 GB on first run, then trains and evaluates)
TASKS = 'T1,T2,T3'          # e.g. 'T2' for the cross-cohort task only
OVERRIDES = ''              # e.g. 't2_target_ssl=false ssl_steps=1000 task_steps=1000' to run faster
OUT = f'{WORK}/results/full'
!python scripts/run_benchmark.py --profile full --data-root {WORK}/data --scgpt-dir {WORK}/models/scGPT-human --out {OUT} --tasks {TASKS} --set {OVERRIDES} 2>&1 | grep -vi warn

In [ ]:
#@title 6 · Results: the PROMISE table and the full report
import pandas as pd
from IPython.display import Markdown, display
p = f'{OUT}/PROMISE.csv'
if os.path.exists(p):
    display(pd.read_csv(p).round(3))
print(open(f'{OUT}/RECOMMENDED_MODEL.txt').read() if os.path.exists(f'{OUT}/RECOMMENDED_MODEL.txt') else '')
display(Markdown(open(f'{OUT}/REPORT.md').read()))

In [ ]:
#@title 7 · Use the shipped model on new cells
from amladapter.model import AMLAdapterModel
from amladapter.data import load_naldini
model = AMLAdapterModel.load(f'{OUT}/aml_adapter_final.pt', scgpt_dir=f'{WORK}/models/scGPT-human', device='cuda')
nd = load_naldini(f'{WORK}/data')                 # cached from step 5
demo = nd[nd.obs.stage.eq('post').to_numpy()][:2000]
pred = model.predict(demo)
display(pred.join(demo.obs[['patient', 'timepoint', 'malignant']]).head(10))
from sklearn.metrics import roc_auc_score
lab = demo.obs.malignant.notna().to_numpy()
print('AUROC vs NPM1 genotype on these post-treatment cells:', round(roc_auc_score(demo.obs.malignant[lab], pred.malignant_prob[lab]), 3))
# model.embed(adata) returns AML-adapted embeddings for UMAP / clustering / your own classifiers